In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

class SubtractOne(nn.Module):
  def forward(self, img): # Should be mask instead of img
    return img-1

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Resize((28, 28)),
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor
    transforms.ToTensor(),
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225]),
    SubtractOne()
])


# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
import os
import random
import numpy as np
import torch
# import torch.nn as ##
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
# from ##  import datasets, transforms
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.datasets import MNIST
import os
import random
import numpy as np
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt


In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# letters = ['A', 'B']

# Create DataLoaders and display samples
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True, num_workers = 2)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = False, num_workers = 2)

# Write your code here

fig, axes = plt.subplots(2, 6, figsize=(10, 4))
for ax in axes.ravel():
    idx = np.random.randint(0, len(train_dataset))
    sample = train_dataset[idx]
    x, y = sample

    # unnormalize for display (approx)
    x_vis = (x * 0.5) + 0.5
    x_vis = x_vis.permute(1, 2,0)
    ax.imshow(x_vis.squeeze(0), cmap="gray")
    ax.set_title(letters[y], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()




In [ ]:
import torchvision
import torch
import torch.nn as nn
from torchvision.models import efficientnet_v2_s
import torchvision.models as models

# Write your code here
model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1) # Or we can use Weights.DEFAULT to not overthink it

for param in model.features.parameters():
    param.requires_grad = False




# The model currently has a classifier with 1000 out features, I'll adjust that to 26

model.classifier[1] = nn.Linear(model.classifier[1].in_features, 26)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model

In [ ]:
# Write your code here
# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        # images = batch['image'].to(device)
        # labels = batch['label'].to(device)


        outputs = model(images).to(device)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # Track accuracy
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)  # Get class with highest probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # Compute accuracy
            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)  # Get predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy



In [ ]:
# Write your code here
# Training setup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# I am getting this weird error and I have no idea what it is, I think everything I did was correct but I'm still getting this

criterion = nn.CrossEntropyLoss()

# TO-DO: Set learning rate
learning_rate = 0.0001


optimizer = optim.Adam(
    model.classifier.parameters(),
    lr=learning_rate
)


num_epochs = 10  # How many times to iterate through the dataset?

# Initialize history tracking
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training loop
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(test_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(test_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')


In [ ]:
# Write your code here
